# Datamine SURTRI Process Example

This notebook demonstrates how to configure and run the **`surtri`** process wrapper in `dmstudio`.

## Process Description

## SURTRI

Generate a triangulated digital terrain model from perimeter, string and/or point data subject to optional boundary and string edge constraints.

Points, strings, perimeters and boundaries may be input to the process. Additional points may be added into strings. Crossing strings are automatically resolved. Points duplicated within a tolerance are removed. Maximum separation of points to be joined may be specified. Boundaries may be 2D or 3D internal or external. The centre of gravity of each triangle may be output. An error trace file containing a dump of data may be produced to help locate bad data.

**Note** : Input files for **SURTRI** must not exceed 60,000 strings.

The points used in the tessellation are output into a wireframe points file &**WIREPT** with fields **XP, YP, ZP** and an additional field PID which is the point number allocated by **SURTRI**. The triangle definitions are output into a wireframe triangle file &**WIRETR** with fields **PID1, PID2** and **PID3** which identify the points at each vertex. Additional fields **XBAR, YBAR** and **ZBAR** defining the centre of gravity of each triangle may optionally be added to the triangle file. These output files may be used as input to other wireframing processes such as [ADDTRI](<addtri.md>), [TRIFIL](<trifil.md>), [PLOTWS](<plotws.md>), [ISOTRI](<isotri.md>), [PLOTTR](<plottr.md>), [TRICON](<tricon.md>), [TRIVOL](<trivol.md>), [WIREPE](<wirepe.md>).

### Boundaries

The tessellation may be constrained by a number of bounding perimeters. These perimeters are input via the &PERIMIN file. Bounding perimeters may form either internal or external boundaries. External boundaries will restrict tessellation to within the perimeter while internal boundaries will restrict tessellation to outside the perimeter.

A combination of internal and external boundaries may be used at one time subject to the following rules:

* Boundaries must not cross each other.
* If internal and external boundaries are both present then each internal boundary must be fully contained within an external boundary.

Boundary perimeters may be treated as 2D cookie cutters (@**SYSTEM** =2) or as 3D edges to be included in the tessellation (@**SYSTEM** =3). If 2D boundaries are used then the tessellation will be confined to points within the perimeter. If no external boundaries are used then the tessellation will be confined outside any internal boundaries and within the convex hull of all points.

Perimeters boundary type may be set using an optional * **BOUNDARY** field on the &**PERIMIN** file or via the @**BOUNDARY** parameter.

The **BOUNDARY** parameter may have the values:

- |  missing (use @BOUNDARY),
---|---
0 |  edge constraint only,
1 |  external boundary or
2 |  internal boundary.

All perimeters in the &**PERIMIN** file are treated as closed polygons. They may or may not be explicitly closed with the first and last points being the same.

### Edges

Edge constraints may be input to the tessellation as open or closed strings via the &**STRING** file or as closed perimeters from the &**PERIMIN** file. All strings should be 3D as they are included in the tessellation as points. Perimeters from &**PERIMIN** are included as edges if @**SYSTEM** = 3.

The resultant tessellation will honor all string and perimeter edges provided edges do not cross. Where an edge crosses another edge priority will be given arbitrarily to one edge. Where an edge crosses a bounding edge an additional point will be inserted into the string just inside the boundary. The distance from this point to the boundary is determined by the height to base ratio of the triangle formed by the 2 boundary edge points and the new point. This ratio may be controlled using the @**HBRATIO** parameter. In the vast majority of cases the default value (0.05) should suffice.

Strings and perimeters therefore may cross over each other and any boundaries.

Additional points may be inserted into strings to improve the tessellation. The maximum separation of points inserted is defined by the @DMAX parameter.

### Points

All points input to the tessellation including those from &**POINTIN** as well as string points, 3D boundary points and additional points created using @**DMAX** are filtered for duplicates using a finite tolerance. The default value for this (0.001) may be overwritten using the @**TOL** parameter however the actual tolerance used may be increased by the program. The minimum accepted tolerance is set to the maximum range of the data (**XMAX-XMIN** or **YMAX-YMIN**) divided by 500000. The tolerance used is written to the screen.

### Partitions

The maximum number of points in total that may be passed to the Delaunay algorithm is limited by the size of the real array compiled into your application. Typically this is 500000 bytes on a workstation or 100000 bytes on a PC. The maximum number of points is ( bytes/20 -3 ). For 500000 bytes this equates to 24297 points. If this value is exceeded **SURTRI** will automatically partition the data into multiple passes. The maximum number of points per pass may be reduced using the @**MAXPTS** parameter (this should only be required for testing the process).

**SURTRI** partitions according to sub-ranges of X or Y depending on which has the largest overall range. Each partition will have a Delaunay optimum tessellation, however the process of knitting together partitions may not be optimum. Strings crossing partitions will be honored. If **SURTRI** goes into partitioning some points may be duplicated in the output points file although this will not cause any problems with the model.

### Trace file

**SURTRI** has the option of outputting a file, **SURTRI.TRC** containing a dump of the point and edge data passed to the tessellation algorithm for each pass. This file may also contain a set of plot commands that may be interpreted by the **MINSURV** program. The amount of trace output is controlled by the @**ERRTRACE** parameter.

The tessellation program will return an error status for each pass, = 0 no error = 1 warning = 2 for a serious error. Normally if a serious error is encountered the process will stop and no points or triangles will be output. Setting @**ERRTRACE** will force the output of data despite a serious error. Triangles produced in this way should not be used in subsequent processing.

The SURTRI.TRC file may be used to interpret error messages from the tessellation program. It contains a list of points and edges with their co-ordinates. Error messages from the routines **DELAUN, EDGE, BCLIP** and **TCHECK** may refer to point numbers and edges in this file.

### Errors in triangulation

During a triangulation pass a number of error messages may appear from the following routines.

## ***ERROR IN DELAUN***

## ***ERROR IN BCLIP***

## ***ERROR IN EDGE***

## ***ERROR IN TCHECK***

At the end of each triangulation pass an error status flag will be reported. This flag has the values:

* 0 no error.

* 1 warning.

* 2 serious.

The error messages will usually give a reference to point or edge numbers. The coordinates of these may be checked by running **SURTRI** with @**ERRTRACE** =1 and reading the output SURTRI.TRC file.

Errors with a status of 1 usually refer to crossing strings and may be ignored.

Serious errors are likely caused by duplicate points that have not been filtered. try increasing the tolerance using the @TOL parameter.

### Input Files:

* **perimin** (String):
  Input perimeter file containing **XP,YP,ZP,PTN, PVALUE** and optional **BOUNDARY**
  fields. Perimeters may be included in the triangulation and/or used as boundaries.ALL
  are assumed closed. The **BOUNDARY** field may have the values : - missing (use
  **BOUNDARY**) , 0 edge constraint, 1 external boundary or 2 internal boundary.
  Required=No

* **string** (String):
  Input string file containing **XP,YP,ZP,PTN** and **PVALUE** fields. String segments
  are included in the triangulation as 3D edge constraints, breaklines. Strings may be
  open or closed.
  Required=No

* **pointin** (Point Data):
  Input point file containing **XPT,YPT,ZPT** fields.
  Required=No

### Output Files:

* **wiretr** (Wireframe Triangle):
  Output wireframe triangle file. May include additional **XBAR,YBAR** and **ZBAR**
  fields.
  Required=Yes

* **wirept** (Wireframe Points):
  Output wireframe point file.
  Required=Yes

### Fields:

* **xpt** (Numeric : POINTIN):
  X field in input point file.
  Default=Undefined
  Required=No

* **ypt** (Numeric : POINTIN):
  Y field in input point file.
  Default=Undefined
  Required=No

* **zpt** (Numeric : POINTIN):
  Z field in input point file.
  Default=Undefined
  Required=No

### Parameters:

* **surface**:
  Optional surface identifier, +1 for upper surface, -1 for lower (1).
  Range=-1, 1
  Values=-1,1
  Default=1
  Required=No

* **boundary**:
  Default boundary specifier for perimeters. Used if the **BOUNDARY** field does not
  exist in **PERIMIN** or has a missing value (0). 0 edge constraints, must be 3D, 1
  external boundary or 2 internal boundary.
  Range=0,2
  Values=0,1,2
  Default=0
  Required=No

* **system**:
  Defines the treatment of boundary perimeters **BOUNDARY** = 1 or 2 from **PERIMIN**
  (3). 2 perimeters are 2D and used only as constraints. 3 perimeters are 3D and are
  included in the triangulation.
  Range=2,3
  Values=2,3
  Default=3
  Required=No

* **dmax**:
  The maximum separation of additional points interpolated into long string segments to
  improve the triangulation.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **maxlink**:
  Maximum separation of points that will be joined by a triangle.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **tol**:
  Tolerance distance below which points are considered to be duplicated. If too small
  this value may automatically be increased by the program.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **cog**:
  Used to include extra fields containing the coordinates of the centre of gravity of
  each triangle in output triangle file (0): Options: 0: \- do not include **XBAR,YBAR**
  or **ZBAR** .; 1: \- include **ZBAR** only.; 2: \- include **XBAR,YBAR**
  Range=0,2
  Values=0,1,2
  Default=0
  Required=No

* **errtrace**:
  Used to control the amount of error messages and the creation of a system file
  SURTRI.TRC containing a dump of points and edges for tracing errors. Also may be used to
  force the creation of output points and triangle records despite a serious error(0).

* **Options** (0: \- no error trace, reduced error messages.; 1: \- create **SURTRI.TRC** ,):
  force output, full error reporting.; 2: \- as 1 plus include **MINSURV** plot records.
  Range=0,2
  Values=0,1,2
  Default=0
  Required=No

* **hbratio**:
  Height to base ratio of triangles created to resolve crossing strings (0.05).
  Range=Undefined
  Values=Undefined
  Default=0.05
  Required=No

* **maxpts**:
  Overwrite the program limitation on maximum points per partition with a smaller value.
  May be used to force partitioning for testing (-).
  Range=Undefined
  Values=Undefined
  Default=-
  Required=No

* **check**:
  Range=0,1
  Values=0,1
  Default=1
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('surtri')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute surtri
print("Running surtri...")
dm_cmd.surtri(
    string_i='t_SurfaceTriangles',  # required
    pointin_i='t_SurfacePointsPt',  # required
    wiretr_o='t_surtri_out',  # required
    wirept_o='t_surtri_out',  # required
    # perimin_i='optional',  # optional
    # xpt_f='optional',  # optional
    # ypt_f='optional',  # optional
    # zpt_f='optional',  # optional
    # surface_p=1,  # optional
    # boundary_p=0,  # optional
    # system_p=3,  # optional
    # dmax_p='optional',  # optional
    # maxlink_p='optional',  # optional
    # tol_p='optional',  # optional
    # cog_p=0,  # optional
    # errtrace_p=0,  # optional
    # hbratio_p=0.05,  # optional
    # maxpts_p='-',  # optional
    # check_p=1,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("surtri execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_surtri_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")